EDGELIST GENERATOR: get edgelist for each dataset.

**UPLOAD LIBRARIES**

In [ ]:
#Libraries
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import json
from pandas import json_normalize
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from networkx.drawing.layout import bipartite_layout
from networkx.algorithms import bipartite
import seaborn as sns
import numpy as np

from google.colab import files

def createList(r1, r2):
    return [item for item in range(r1, r2+1)]

In [ ]:
# #Load dataframe from Drive
#Aprile, 1-23
#Marzo: 1-31 (11 e 12 mancano)
#Febbraio: 20-29
month = 4 #feb = 2, march = 3, apr = 4
start_day = 22 #|| 1-23 april
last_day = 23

day_lst = createList(start_day, last_day)

**UPLOAD DATA**

In [ ]:
for day in day_lst:
  df = pd.read_json(f"/content/drive/MyDrive/MOTIFS PAPER/DATASETS/coronavirus_2020_0{month}_{day}.json",lines=True)
  #df = pd.read_json(f"/content/drive/MyDrive/MOTIFS PAPER/DATASETS/coronavirus_2020_0{month}_0{day}.json",lines=True)

  #Take only info we need
  df1 = df[['user', 'retweeted_status']]
  df1 = df[df['retweeted_status'].notna()]

  df1_userobj_raw= pd.json_normalize(df1['user'])
  df1_userobj = df1_userobj_raw[['id_str', 'verified']]
  df1_A = df1_userobj.copy()
  df1_A.rename(columns={"id_str": "user_id_str", "verified": "user_verified"}, inplace=True)

  df1_retweetobj_raw = pd.json_normalize(df1['retweeted_status'])
  df1_retweetobj = df1_retweetobj_raw[['user.id_str', 'user.verified']]
  df1_B = df1_retweetobj.copy()
  df1_B.rename(columns={"user.id_str": "retweet_author_id", "user.verified": "retweet_author_verified"}, inplace=True)

  df2 = pd.concat([df1_A, df1_B], axis = 1)
  df3 = df2[df2['user_verified'] != df2['retweet_author_verified']] #we take only the pair unverified-verified or verified-unverified; ignoring verified-verified and unverified-unverified

  # Caso 1: l'utente che retweeta è verificato (e l'autore è non verificato)
  df_case1 = df3[df3['user_verified'] == True].copy()
  df_case1_renamed = pd.DataFrame({
      'verified': df_case1['user_id_str'],
      'unverified': df_case1['retweet_author_id']
  })
  # Caso 2: autore del retweet è verificato (e l'utente che retweeta è non verificato)
  df_case2 = df3[df3['retweet_author_verified'] == True].copy()
  df_case2_renamed = pd.DataFrame({
      'verified': df_case2['retweet_author_id'],
      'unverified': df_case2['user_id_str']
  })

  df_final_pairs = pd.concat([df_case1_renamed, df_case2_renamed], axis=0)
  df_final_pairs.drop_duplicates(inplace=True)

  df_final_pairs.to_csv(f'/content/drive/MyDrive/MOTIFS PAPER/CODES/Edgelists/edgelist_2020_0{month}_{day}.csv', index=False)
  print(f"Month {month}, Day {day} processed ")

**DATA DOWNLOAD** <br>
Note: rename file before uploading it to Drive.



In [ ]:
#Uncomment this cell to verify the upload
# df_verify = pd.read_csv('/content/drive/MyDrive/MOTIFS PAPER/CODES/Edgelists/edgelist_2020_02_20.csv')
# print(df_verify)

**DA USARE SOLO DOPO AVER PROCESSATO TUTTI I FILE**

In [ ]:
import glob
import os

path = r'/content/drive/MyDrive/MOTIFS PAPER/CODES/Edgelists' # use your path
all_files = glob.glob(os.path.join(path, "*.csv"))
df_all = pd.concat((pd.read_csv(f) for f in all_files), ignore_index=True)

edgelist_final = df_all.drop_duplicates() #final dataset with node id only

In [ ]:
edgelist_final.to_csv(f'/content/drive/MyDrive/MOTIFS PAPER/CODES/Edgelists/edgelist_final.csv', index=False)

In [ ]:
edgelist_final